# 105 — Merge Vector Tiles

Merges all per-tile polygon GeoPackages into a single output file per threshold and MMU size.
Generates centroids from the merged polygons.
Reassigns patch IDs sequentially across all tiles.

Run after `104_vectorize.ipynb`.

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
from config import *
from helpers import *
import time
import gc
import numpy as np

print(f"Config loaded ✓")
print(f"  Mode:       {MODE}")
print(f"  Thresholds: {PIXEL_THRESHOLDS}")
print(f"  MMU sizes:  {MMU_SIZES}")

In [ ]:
# ── Check available vector tiles ──────────────────────────────────────────────
print("── Checking vector tiles ───────────────────────────────────")

for thresh in PIXEL_THRESHOLDS:
    for min_m2, label in MMU_SIZES:
        vec_dir   = vector_tile_dir(thresh, label)
        poly_files = [f for f in os.listdir(vec_dir) if f.endswith(".gpkg") and "_poly_" in f]
        total_size_mb = sum(os.path.getsize(os.path.join(vec_dir, f)) for f in poly_files) / 1e6
        print(f"  t={thresh} min={label}: {len(poly_files)} tiles  ({total_size_mb:.1f} MB)")

In [ ]:
# ── Merge polygon tiles → final output ────────────────────────────────────────
import pandas as pd

print("── Merging polygon tiles ───────────────────────────────────")
suffix = output_suffix()
t0     = time.time()

for thresh in PIXEL_THRESHOLDS:
    for min_m2, label in MMU_SIZES:
        print(f"\n  t={thresh} min={label} …")

        vec_dir    = vector_tile_dir(thresh, label)
        poly_files = sorted(
            os.path.join(vec_dir, f)
            for f in os.listdir(vec_dir)
            if f.endswith(".gpkg") and "_poly_" in f
        )

        if not poly_files:
            print(f"    [!] No polygon tiles found — run 104_vectorize.ipynb first")
            continue

        # Load and concatenate all tile polygons
        gdfs = []
        for p in poly_files:
            try:
                gdf = gpd.read_file(p)
                if len(gdf) > 0:
                    gdfs.append(gdf)
            except Exception as e:
                print(f"    [!] Failed to read {os.path.basename(p)}: {e}")

        if not gdfs:
            print(f"    [!] No valid polygon tiles")
            continue

        merged = gpd.GeoDataFrame(
            pd.concat(gdfs, ignore_index=True),
            crs=gdfs[0].crs
        )
        del gdfs; gc.collect()
        print(f"    Loaded: {len(merged):,} patches from {len(poly_files)} tiles")

        # Dissolve to fix tile boundary gaps
        print(f"    Dissolving (buffer={DISSOLVE_BUFFER_M}m) …")
        merged["_dissolve"] = 1
        merged["geometry"]  = merged.geometry.buffer(DISSOLVE_BUFFER_M)
        merged = merged.dissolve(by="_dissolve").explode(index_parts=False).reset_index(drop=True)
        merged["geometry"]  = merged.geometry.buffer(-DISSOLVE_BUFFER_M)
        merged = merged[merged.geometry.is_valid & ~merged.geometry.is_empty].reset_index(drop=True)
        merged["area_m2"]   = merged.geometry.area.round(2)
        merged["area_ha"]   = (merged["area_m2"] / 10000).round(4)
        merged = merged[merged["area_m2"] >= min_m2].reset_index(drop=True)
        merged["patch_id"]  = merged.index + 1

        print(f"    Patches:    {len(merged):,}")
        print(f"    Total area: {merged['area_ha'].sum():.2f} ha")
        print(f"    Largest:    {merged['area_ha'].max():.4f} ha")
        print(f"    Median:     {merged['area_ha'].median():.4f} ha")

        # Save merged polygons
        poly_out = os.path.join(PP_OUT_DIR,
            f"lupine_polygons_{suffix}_{MODEL_NAME}_min{label}_t{thresh}{shadow_suffix()}.gpkg")
        merged[["patch_id", "area_m2", "area_ha", "geometry"]].to_file(
            poly_out, driver="GPKG")
        print(f"    Polygons  → {os.path.basename(poly_out)}")

        # Generate and save centroids
        cents = merged.copy()
        cents["geometry"] = cents.geometry.centroid
        cents["lupine"]   = True
        cent_out = os.path.join(PP_OUT_DIR,
            f"lupine_centroids_{suffix}_{MODEL_NAME}_min{label}_t{thresh}{shadow_suffix()}.gpkg")
        cents[["patch_id", "area_m2", "area_ha", "lupine", "geometry"]].to_file(
            cent_out, driver="GPKG")
        print(f"    Centroids → {os.path.basename(cent_out)}")

        del merged, cents; gc.collect()

print(f"\n  Done. Time: {(time.time()-t0)/60:.1f}min")

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
import pandas as pd

print("── Summary ──────────────────────────────────────────────────")
suffix = output_suffix()

rows = []
for thresh in PIXEL_THRESHOLDS:
    for min_m2, label in MMU_SIZES:
        poly_out = os.path.join(PP_OUT_DIR,
            f"lupine_polygons_{suffix}_{MODEL_NAME}_min{label}_t{thresh}.gpkg")
        if not os.path.exists(poly_out):
            continue
        gdf = gpd.read_file(poly_out)
        rows.append({
            "Threshold":   thresh,
            "Min patch":   label,
            "Patches":     len(gdf),
            "Area (ha)":   round(gdf["area_ha"].sum(), 2),
        })
        del gdf

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
import geopandas as gpd
import os

folder = r"C:\Users\petra\Documents\NATT_Test\All"

files = [f for f in os.listdir(folder) if f.startswith("lupine_polygons") and f.endswith(".gpkg")]
print(f"Found {len(files)} files:\n")

total = 0
for f in sorted(files):
    path = os.path.join(folder, f)
    gdf  = gpd.read_file(path)
    print(f"  {f}")
    print(f"    Polygons: {len(gdf):,}  Area: {gdf['area_ha'].sum():.2f} ha")
    total += len(gdf)

print(f"\nTotal polygons across all files: {total:,}")

In [ ]:
import geopandas as gpd
import os

folder      = r"C:\Users\petra\Documents\NATT_Test\All"
lupine_path = r"C:\Users\petra\Documents\NATT_Test\data\lupine.gpkg"
grid_path   = r"C:\Users\petra\Documents\NATT_Test\data\grunnkort_loftmynd_nidurhalgeom.shp"

# Load lupine reference and grid
lupine = gpd.read_file(lupine_path, layer="L14.4 Alaskalúpína")
grid   = gpd.read_file(grid_path).to_crs(lupine.crs)

# Fix geometries
lupine["geometry"] = lupine.geometry.make_valid()
lupine_union       = lupine.union_all()

# Find training tile extent
training_tiles  = grid[grid.intersects(lupine_union)].copy()
training_bounds = training_tiles.union_all()

analyst_area_ha = lupine.geometry.area.sum() / 10000
print(f"Training tiles:      {len(training_tiles)}")
print(f"Analyst lupine area: {analyst_area_ha:.2f} ha")
print()

# Load prediction files
files = [f for f in os.listdir(folder)
         if f.startswith("lupine_polygons") and f.endswith(".gpkg")
         and "centroids" not in f]

for f in sorted(files):
    path = os.path.join(folder, f)
    gdf  = gpd.read_file(path).to_crs(lupine.crs)

    # Clip to training tile extent
    clipped  = gdf[gdf.intersects(training_bounds)].copy()
    total_ha = clipped["area_ha"].sum()

    # Fast clip to analyst polygons
    within_gdf = gpd.clip(clipped, lupine)
    within_ha  = within_gdf.geometry.area.sum() / 10000
    outside_ha = total_ha - within_ha

    print(f"  {f}")
    print(f"    Total polygons:   {len(clipped):,}  |  Total area: {total_ha:.2f} ha")
    print(f"    Within analyst:   {within_ha:.2f} ha  ({within_ha/total_ha*100:.1f}%)")
    print(f"    Outside analyst:  {outside_ha:.2f} ha  ({outside_ha/total_ha*100:.1f}%)")
    print(f"    Analyst coverage: {within_ha/analyst_area_ha*100:.1f}% of analyst area detected")
    print()